# Selección evolutiva de características mediante algoritmo genético binario frente al filtrado univariado en clasificación supervisada

## Investigación aplicada · Fronteras de la Computación Blanda

---
* **Línea de investigación:** Tema N.º 9 — Selección evolutiva de características
* **Autores:**
  * Juan José González Ríos
  * Camilo Eduardo Muñoz Albornoz

---

### Resumen del proyecto

Este cuaderno documenta y ejecuta una investigación experimental sobre la **selección evolutiva de características**, una línea que integra la computación evolutiva con el aprendizaje supervisado para abordar problemas de reducción de dimensionalidad en datos tabulares.

El estudio evalúa si un **algoritmo genético binario** —operando bajo un enfoque *wrapper* con máscaras de selección— logra reducir la cantidad de variables del conjunto de datos *Breast Cancer Wisconsin (Diagnostic)* conservando un desempeño predictivo competitivo en **F1 macro**. Para aislar el efecto de la elección de atributos, el experimento mantiene fijo un clasificador de regresión logística y contrasta el método evolutivo contra una **selección univariada** (*filter* mediante prueba ANOVA F) restringida al mismo número de variables, así como contra una línea base que utiliza las 30 características originales.

El flujo de trabajo garantiza la reproducibilidad y evita la fuga de datos mediante particiones estratificadas independientes (entrenamiento, validación y prueba) repetidas en diez semillas, incorporando además mediciones de tiempo de cómputo, estabilidad entre ejecuciones (índice de Jaccard) y un análisis de sensibilidad sobre el peso de penalización por complejidad.

## 1. Planteamiento del problema y fundamento conceptual

La selección evolutiva de características integra dos áreas fundamentales: la **computación evolutiva** y el **aprendizaje supervisado**. Mientras los modelos supervisados aprenden patrones a partir de ejemplos etiquetados, la computación evolutiva aporta mecanismos heurísticos inspirados en la selección natural para explorar espacios de búsqueda combinatorios donde los métodos exactos resultan inviables.

### El problema de la dimensionalidad y la redundancia

En conjuntos de datos tabulares, conservar la totalidad de las variables disponibles no garantiza un mejor desempeño predictivo. La presencia de atributos irrelevantes introduce ruido que favorece el sobreajuste, mientras que las variables altamente correlacionadas aportan información redundante que incrementa la complejidad del modelo sin mejorar su capacidad de generalización.

Evaluar de forma exhaustiva todos los subconjuntos posibles de $d$ características exige analizar $2^d - 1$ combinaciones. Para un conjunto de datos con $d = 30$ variables, el espacio de búsqueda asciende a $1{,}073{,}741{,}823$ subconjuntos, lo cual hace impráctica la fuerza bruta y justifica el uso de estrategias heurísticas de selección.

### Enfoques *filter* frente a *wrapper*

Para reducir la dimensionalidad manteniendo las variables originales se distinguen principalmente dos familias de métodos:

* **Métodos de filtrado (*filter*):** evalúan cada característica de manera independiente mediante criterios estadísticos univariados (como la prueba ANOVA F) antes de entrenar el clasificador. Son rápidos y deterministas, pero ignoran las interacciones entre variables y pueden retener atributos redundantes si todos presentan puntajes individuales altos.
* **Métodos envoltorio (*wrapper*):** evalúan subconjuntos completos entrenando y validando directamente el clasificador elegido. Un algoritmo genético binario opera dentro de esta familia al codificar cada candidato como una máscara de ceros y unos, permitiendo capturar sinergias y descartar redundancias a cambio de un mayor costo computacional y de variabilidad estocástica entre ejecuciones.

### Pregunta de investigación e hipótesis

* **Pregunta de investigación:** Al mantener fijo un clasificador de regresión logística sobre el conjunto de datos *Breast Cancer Wisconsin (Diagnostic)*, ¿logra un algoritmo genético binario reducir la cantidad de variables conservando un F1 macro en prueba comparable o superior al de la selección univariada con igual número de variables y al del modelo con las 30 características?
* **Hipótesis:** El algoritmo genético binario identifica subconjuntos significativamente más compactos que el conjunto completo y obtiene un F1 macro competitivo o superior frente al filtrado univariado con el mismo número de atributos, al aprovechar combinaciones entre variables; no obstante, requiere un tiempo de cómputo considerablemente mayor y exhibe cierta dispersión en las variables elegidas entre distintas semillas.

### Alcance del estudio

Este cuaderno delimita el experimento a un único conjunto de datos público, un clasificador fijo y una búsqueda evolutiva controlada. La reducción de variables sin pérdida de desempeño se trata como un resultado empírico por verificar y no como una conclusión garantizada.

## 2. Diseño y protocolo experimental

Para asegurar una comparación rigurosa y reproducible, el experimento fija todas las condiciones de evaluación antes de ejecutar las corridas finales.

### Conjunto de datos y clasificador fijo

* **Datos:** *Breast Cancer Wisconsin (Diagnostic)*, cargado directamente desde `scikit-learn` (origen: UCI Machine Learning Repository, licencia CC BY 4.0). Contiene 569 observaciones, 30 variables numéricas continuas y 2 clases (maligno y benigno).
* **Clasificador fijo:** Regresión Logística (`LogisticRegression` con `max_iter=1000` y `random_state=42`) integrada en un `Pipeline` junto con estandarización (`StandardScaler`). Mantener inalterado el clasificador en todos los experimentos garantiza que las variaciones en desempeño respondan exclusivamente al subconjunto de variables seleccionado.

### Métodos comparados

* **Método principal (Algoritmo Genético Binario):** representa cada solución como una máscara binaria $m \in \{0, 1\}^{30}$, donde cada bit indica la inclusión o exclusión de una característica. Emplea selección por torneo (tamaño 3), cruce uniforme (probabilidad $0.8$), mutación *bit-flip* (probabilidad $0.05$ por gen) y elitismo de un individuo por generación. El algoritmo maximiza en el conjunto de validación la siguiente función de aptitud:

$$\text{Fitness}(m) = (1 - \alpha)\cdot \text{F1}_{\text{macro, val}}(m) - \alpha \cdot \frac{\sum_{i=1}^{d} m_i}{d}$$

donde $d = 30$ es el total de variables y $\alpha = 0.05$ es el peso base que penaliza la complejidad del subconjunto. Se garantiza además que ninguna máscara quede completamente vacía.
* **Método de referencia (Selección univariada):** `SelectKBest` con la función estadística `f_classif` (ANOVA F). En cada semilla, el filtro selecciona exactamente el mismo número $k$ de características obtenido por el algoritmo genético, garantizando una comparación justa en igualdad de dimensionalidad.
* **Línea base contextual (Todas las variables):** el mismo clasificador fijo entrenado y evaluado utilizando las 30 características disponibles.

### Métricas, presupuesto y análisis de sensibilidad

* **Métrica principal:** F1 macro sobre el conjunto de prueba (dirección de mejora: maximizar), elegida para ponderar por igual ambas clases.
* **Métricas secundarias:** número de variables seleccionadas (minimizar), tiempo de ejecución en segundos (minimizar) y estabilidad entre ejecuciones medida mediante el índice de Jaccard promedio entre todos los pares de máscaras ganadoras obtenidas en las distintas semillas (maximizar).
* **Presupuesto y repeticiones:** el experimento ejecuta 10 semillas independientes (`0` a `9`). En cada semilla, el algoritmo genético corre con una población de 30 individuos durante 30 generaciones (máximo 900 evaluaciones de validación, optimizadas con caché de máscaras repetidas). Los métodos de referencia requieren una única evaluación por semilla.
* **Prevención de fuga de datos:** en cada semilla se genera una partición estratificada de 60 % entrenamiento, 20 % validación y 20 % prueba. El escalado y el cálculo de puntajes univariados se ajustan únicamente con entrenamiento; la función de aptitud del algoritmo genético se evalúa exclusivamente sobre validación; el conjunto de prueba se reserva para la medición final una vez elegida la máscara.
* **Análisis de sensibilidad acotado:** manteniendo fijos el dataset, el clasificador, las semillas y los operadores evolutivos, el experimento varía únicamente el parámetro de penalización $\alpha \in \{0.01, 0.05, 0.15\}$ para cuantificar el compromiso entre desempeño predictivo y reducción de variables.
* **Riesgo técnico y mitigación:** el costo acumulado de repetir diez semillas y tres niveles de sensibilidad podría elevar el tiempo de ejecución; el uso de memoria caché para máscaras repetidas y un límite acotado de iteraciones en la regresión logística mantienen la ejecución viable en un computador convencional.

## 3. Revisión de literatura y síntesis crítica

La fundamentación teórica del estudio se apoya en cinco fuentes académicas primarias arbitradas: dos investigaciones recientes (2025–2026) que muestran el estado actual de la selección evolutiva de características y tres trabajos de referencia sobre taxonomía de selección de variables, computación evolutiva y medición de estabilidad.

#### Fuente 1 — Keivanian et al. (2026)
* **Referencia y enlace:** Keivanian, F., Shafiabady, N., Noman, N., Fan, Z., & Mirjalili, S. (2026). *A fuzzy multi-objective neuro-evolutionary framework with bargaining-based selection for interpretable body fat prediction*. Neurocomputing, 675, 132974. https://doi.org/10.1016/j.neucom.2026.132974
* **Año y estado:** 2026 — Artículo arbitrado de revista.
* **Problema y método:** Aborda el equilibrio entre precisión predictiva, simplicidad e interpretabilidad mediante un marco neuroevolutivo multiobjetivo difuso con selección basada en negociación.
* **Datos y comparación:** Se evalúa en predicción de grasa corporal frente a modelos alternativos de aprendizaje automático y esquemas de selección convencionales.
* **Hallazgo y limitación:** Evidencia que optimizar simultáneamente desempeño y complejidad produce soluciones más interpretables sin degradar de forma severa el error; sin embargo, la arquitectura híbrida introduce una alta complejidad algorítmica y un elevado número de hiperparámetros que dificultan su replicación en entornos acotados.
* **Relación con la pregunta de investigación:** Funciona como antecedente metodológico reciente del compromiso entre precisión y simplicidad que este cuaderno estudia en una versión controlada mediante una función de aptitud ponderada.

#### Fuente 2 — Song et al. (2025)
* **Referencia y enlace:** Song, X., Ma, H., Zhang, Y., Gong, D., Guo, Y., & Hu, Y. (2025). *A Streaming Feature Selection Method Based on Dynamic Feature Clustering and Particle Swarm Optimization*. IEEE Transactions on Evolutionary Computation, 29(5), 1888–1902. https://doi.org/10.1109/TEVC.2024.3451688
* **Año y estado:** 2025 — Artículo arbitrado de revista.
* **Problema y método:** Estudia la selección de características en escenarios donde las variables llegan de forma continua, combinando agrupamiento dinámico de atributos con optimización por enjambre de partículas (PSO) para descartar variables irrelevantes y redundantes.
* **Datos y comparación:** Experimenta sobre 12 conjuntos de datos de distinta dificultad y un caso de aplicación real, comparando su desempeño y tiempo frente a algoritmos existentes de selección en flujo.
* **Hallazgo y limitación:** Demuestra mejoras en calidad de clasificación dentro de tiempos razonables al controlar la redundancia mediante agrupamiento; no obstante, sus resultados dependen de un escenario dinámico de llegada secuencial de atributos y requieren mecanismos adicionales que exceden el alcance de datos tabulares estáticos pequeños.
* **Relación con la pregunta de investigación:** Refuerza la vigencia de las metaheurísticas poblacionales para combatir la redundancia entre características y justifica evaluar el costo computacional junto con el desempeño predictivo.

#### Fuente 3 — Xue et al. (2016)
* **Referencia y enlace:** Xue, B., Zhang, M., Browne, W. N., & Yao, X. (2016). *A Survey on Evolutionary Computation Approaches to Feature Selection*. IEEE Transactions on Evolutionary Computation, 20(4), 606–626. https://doi.org/10.1109/TEVC.2015.2504420
* **Año y estado:** 2016 — Artículo de revisión arbitrado en revista.
* **Problema y método:** Sistematiza los enfoques de computación evolutiva aplicados a selección de características en clasificación (algoritmos genéticos, PSO, programación genética y colonia de hormigas), diferenciando representaciones binarias, funciones objetivo de un solo criterio con penalización y formulaciones multiobjetivo.
* **Datos y comparación:** Revisa de manera transversal literatura experimental en clasificación, minería de datos y selección de biomarcadores.
* **Hallazgo y limitación:** Concluye que la búsqueda global de la computación evolutiva supera frecuentemente a métodos voraces al capturar interacciones entre variables, pero advierte dos limitaciones persistentes: el alto costo de cómputo de los esquemas *wrapper* y el riesgo de sobreajuste si no se separa estrictamente la validación del conjunto de prueba.
* **Relación con la pregunta de investigación:** Fundamenta la elección de la representación mediante máscara binaria, la función de aptitud que combina F1 y tamaño del subconjunto, y el protocolo estricto de prevención de fuga de datos adoptado en este cuaderno.

#### Fuente 4 — Guyon y Elisseeff (2003)
* **Referencia y enlace:** Guyon, I., & Elisseeff, A. (2003). *An Introduction to Variable and Feature Selection*. Journal of Machine Learning Research, 3(Mar), 1157–1182. https://www.jmlr.org/papers/v3/guyon03a.html
* **Año y estado:** 2003 — Artículo arbitrado fundacional en revista.
* **Problema y método:** Establece el marco conceptual clásico de la selección de variables, definiendo las diferencias estructurales entre métodos *filter* (como el ordenamiento univariado), *wrapper* y *embedded*.
* **Datos y comparación:** Ilustra mediante ejemplos analíticos y aplicaciones en clasificación de texto y microarreglos cómo afectan la correlación, la redundancia y la complementariedad entre variables.
* **Hallazgo y limitación:** Demuestra que variables individuales con bajo poder discriminativo pueden ser altamente informativas al combinarse, y que variables muy correlacionadas no necesariamente aportan ganancia adicional; sin embargo, al ser un marco introductorio, no aborda las metaheurísticas poblacionales modernas ni sus compromisos de cómputo actuales.
* **Relación con la pregunta de investigación:** Justifica teóricamente la comparación directa entre el algoritmo genético (*wrapper*) y la selección univariada ANOVA F (*filter*) bajo igualdad en el número de características.

#### Fuente 5 — Nogueira, Sechidis y Brown (2018)
* **Referencia y enlace:** Nogueira, S., Sechidis, K., & Brown, G. (2018). *On the Stability of Feature Selection Algorithms*. Journal of Machine Learning Research, 18(174), 1–54. https://jmlr.org/papers/v18/17-514.html
* **Año y estado:** 2018 — Artículo arbitrado en revista.
* **Problema y método:** Analiza la estabilidad de los algoritmos de selección de características entendida como la consistencia de los subconjuntos elegidos frente a variaciones de muestreo y naturaleza estocástica, formalizando propiedades estadísticas deseables sobre matrices binarias de selección.
* **Datos y comparación:** Evalúa y contrasta medidas clásicas de similitud entre conjuntos (como Jaccard y Kuncheva) y propone un estimador estadístico con pruebas de hipótesis sobre conjuntos de datos de referencia.
* **Hallazgo y limitación:** Señala que evaluar únicamente la precisión predictiva resulta insuficiente porque algoritmos estocásticos o con datos redundantes pueden elegir subconjuntos muy distintos con desempeño similar; también advierte que medidas simples como Jaccard pueden verse influidas por el tamaño del subconjunto seleccionado.
* **Relación con la pregunta de investigación:** Sustenta la inclusión de la estabilidad entre las diez semillas como métrica secundaria del experimento y aporta cautela crítica al interpretar el índice de Jaccard cuando existen atributos altamente correlacionados.

---

Las cinco fuentes coinciden en que la selección de características persigue un compromiso inevitable entre **desempeño predictivo** y **simplicidad del subconjunto**. Mientras Guyon y Elisseeff (2003) establecen por qué los filtros univariados fallan al ignorar sinergias y redundancias entre atributos, Xue et al. (2016) muestran cómo la computación evolutiva aborda ese espacio combinatorio mediante búsqueda global poblacional. Los trabajos recientes de Song et al. (2025) y Keivanian et al. (2026) confirman que la línea sigue activa, desplazándose hacia esquemas híbridos, multiobjetivo y de manejo dinámico de redundancia.

No obstante, entre los estudios existen diferencias sustanciales en escala y condiciones experimentales. Las propuestas recientes operan con arquitecturas complejas, flujos dinámicos de variables o marcos neurodifusos con múltiples etapas que no permiten generalizar automáticamente sus ganancias a problemas tabulares pequeños con presupuesto reducido. Además, tanto Xue et al. (2016) como Nogueira et al. (2018) subrayan dos riesgos que suelen quedar opacados por el entusiasmo hacia la precisión: el costo computacional de los métodos *wrapper* y la inestabilidad de los subconjuntos elegidos cuando varias columnas contienen información casi equivalente.

En consecuencia, este cuaderno delimita una pregunta empírica acotada y verificable: evaluar, bajo un protocolo estricto sin fuga de datos y con un clasificador fijo, si un algoritmo genético binario sencillo justifica su mayor costo de cómputo frente a un filtro univariado clásico en un conjunto de datos con alta redundancia, midiendo simultáneamente F1 macro, número de variables, tiempo y estabilidad entre ejecuciones.

## 4. Implementación experimental y prueba piloto

### Instalación de dependencias

Esta celda instala de forma silenciosa las bibliotecas científicas requeridas para la ejecución del experimento (`numpy`, `scikit-learn` y `matplotlib`). En entornos que ya disponen de estos paquetes, el gestor verifica las instalaciones existentes sin alterar el flujo del cuaderno.
Tambiénimporta los módulos estándar y científicos del proyecto y define la estructura `Configuracion`, la cual centraliza el identificador del tema, los autores, las diez semillas de reproducibilidad, la métrica principal, las proporciones de partición y los hiperparámetros del algoritmo genético binario, incluidos los valores de penalización $\alpha$ para el estudio de sensibilidad. Al ejecutarse, imprime un resumen de la configuración activa y las versiones exactas del entorno.

In [6]:
%pip install -q numpy scikit-learn matplotlib

from dataclasses import dataclass, field
import sys
import time
import numpy as np
import sklearn
import matplotlib
import pandas as pd
from pathlib import Path
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

@dataclass
class Configuracion:
    tema: int = 9
    integrantes: tuple[str, str] = (
        "Juan José González Ríos",
        "Camilo Eduardo Muñoz Albornoz",
    )
    semillas: list[int] = field(default_factory=lambda: list(range(10)))
    metrica_principal: str = "f1_macro"
    minimizar: bool = False
    tamano_poblacion: int = 30
    generaciones: int = 30
    tamano_torneo: int = 3
    prob_cruce: float = 0.8
    prob_mutacion: float = 0.05
    alpha_base: float = 0.05
    alphas_sensibilidad: tuple[float, ...] = (0.01, 0.05, 0.15)
    proporcion_val: float = 0.20
    proporcion_prueba: float = 0.20

config = Configuracion()

print(f"Tema seleccionado: {config.tema}")
print(f"Integrantes: {config.integrantes[0]} y {config.integrantes[1]}")
print(f"Métrica principal: {config.metrica_principal} (minimizar={config.minimizar})")
print(f"Semillas configuradas: {len(config.semillas)}")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__} | scikit-learn: {sklearn.__version__} | Matplotlib: {matplotlib.__version__}")


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Tema seleccionado: 9
Integrantes: Juan José González Ríos y Camilo Eduardo Muñoz Albornoz
Métrica principal: f1_macro (minimizar=False)
Semillas configuradas: 10
Python: 3.12.6
NumPy: 2.5.5 | scikit-learn: 1.9.1 | Matplotlib: 3.11.2


### Carga del conjunto de datos y partición estratificada

Esta celda define dos funciones: `cargar_datos`, que obtiene la matriz de características, el vector de etiquetas y los nombres de las 30 variables del conjunto *Breast Cancer Wisconsin (Diagnostic)*; y `dividir_datos`, que divide los registros en subconjuntos estratificados de 60 % entrenamiento, 20 % validación y 20 % prueba según una semilla específica. La separación estricta garantiza que el conjunto de prueba permanezca aislado durante todo el proceso de selección.

In [7]:
def cargar_datos():
    conjunto = load_breast_cancer()
    return conjunto.data, conjunto.target, list(conjunto.feature_names)

def dividir_datos(X, y, semilla, proporcion_val=0.20, proporcion_prueba=0.20):
    X_temp, X_prueba, y_temp, y_prueba = train_test_split(
        X,
        y,
        test_size=proporcion_prueba,
        stratify=y,
        random_state=semilla,
    )
    proporcion_val_relativa = proporcion_val / (1.0 - proporcion_prueba)
    X_entrenamiento, X_val, y_entrenamiento, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=proporcion_val_relativa,
        stratify=y_temp,
        random_state=semilla,
    )
    return X_entrenamiento, X_val, X_prueba, y_entrenamiento, y_val, y_prueba

X_total, y_total, nombres_variables = cargar_datos()
clases, conteos = np.unique(y_total, return_counts=True)
print(f"Observaciones: {X_total.shape[0]} | Variables: {X_total.shape[1]}")
print(f"Distribución de clases: clase 0 = {conteos[0]}, clase 1 = {conteos[1]}")

Observaciones: 569 | Variables: 30
Distribución de clases: clase 0 = 212, clase 1 = 357


### Clasificador fijo, métodos de referencia y estabilidad

Esta celda implementa las funciones base compartidas por todos los métodos del experimento:

* `crear_clasificador` construye el `Pipeline` fijo compuesto por estandarización (`StandardScaler`) y regresión logística (`LogisticRegression`), evitando fuga de datos al ajustar el escalado únicamente sobre el conjunto de entrenamiento.
* `evaluar_mascara` entrena el clasificador fijo con las columnas activas de una máscara binaria y calcula el F1 macro sobre el conjunto de evaluación indicado.
* `evaluar_todas_variables` ejecuta la línea base contextual utilizando las 30 características.
* `evaluar_seleccion_univariada` ajusta el filtro `SelectKBest` (`f_classif`) únicamente sobre entrenamiento para seleccionar exactamente $k$ variables y evaluar el clasificador fijo en prueba.
* `calcular_estabilidad_jaccard` calcula el índice de Jaccard promedio entre todos los pares de máscaras binarias obtenidas a lo largo de múltiples semillas.

In [8]:
def crear_clasificador():
    return Pipeline([
        ("escalador", StandardScaler()),
        ("clasificador", LogisticRegression(max_iter=1000, random_state=42)),
    ])

def evaluar_mascara(X_entrenamiento, y_entrenamiento, X_eval, y_eval, mascara):
    modelo = crear_clasificador()
    modelo.fit(X_entrenamiento[:, mascara], y_entrenamiento)
    predicciones = modelo.predict(X_eval[:, mascara])
    return float(f1_score(y_eval, predicciones, average="macro"))

def evaluar_todas_variables(X_entrenamiento, y_entrenamiento, X_prueba, y_prueba):
    inicio = time.perf_counter()
    mascara_completa = np.ones(X_entrenamiento.shape[1], dtype=bool)
    f1_prueba = evaluar_mascara(
        X_entrenamiento, y_entrenamiento, X_prueba, y_prueba, mascara_completa
    )
    tiempo_s = time.perf_counter() - inicio
    return {
        "mascara": mascara_completa,
        "f1_prueba": f1_prueba,
        "num_variables": int(np.sum(mascara_completa)),
        "tiempo_s": tiempo_s,
        "evaluaciones": 1,
    }

def evaluar_seleccion_univariada(X_entrenamiento, y_entrenamiento, X_prueba, y_prueba, k):
    inicio = time.perf_counter()
    selector = SelectKBest(score_func=f_classif, k=k)
    selector.fit(X_entrenamiento, y_entrenamiento)
    mascara = selector.get_support()
    f1_prueba = evaluar_mascara(
        X_entrenamiento, y_entrenamiento, X_prueba, y_prueba, mascara
    )
    tiempo_s = time.perf_counter() - inicio
    return {
        "mascara": mascara,
        "f1_prueba": f1_prueba,
        "num_variables": int(np.sum(mascara)),
        "tiempo_s": tiempo_s,
        "evaluaciones": 1,
    }

def calcular_estabilidad_jaccard(mascaras):
    n = len(mascaras)
    if n < 2:
        return 1.0
    similitudes = []
    for i in range(n):
        for j in range(i + 1, n):
            interseccion = int(np.sum(mascaras[i] & mascaras[j]))
            union = int(np.sum(mascaras[i] | mascaras[j]))
            similitudes.append(interseccion / union if union > 0 else 0.0)
    return float(np.mean(similitudes))

### Método principal: algoritmo genético binario

Esta celda implementa el algoritmo evolutivo encargado de buscar subconjuntos óptimos de características. Cada individuo es una máscara booleana de longitud $d = 30$ con la restricción de contener al menos una variable activa (`asegurar_mascara_no_vacia`).

Durante la evolución, el algoritmo utiliza un diccionario de caché indexado por los bytes de cada máscara para evitar reevaluar combinaciones repetidas. En cada generación aplica elitismo (conservando el mejor individuo intacto), selección por torneo, cruce uniforme y mutación *bit-flip*. La función de aptitud se calcula exclusivamente sobre el conjunto de validación combinando F1 macro y penalización por complejidad $\alpha$. Al finalizar las generaciones, la mejor máscara encontrada se evalúa una única vez sobre el conjunto de prueba.

In [9]:
def asegurar_mascara_no_vacia(mascara, rng):
    if not np.any(mascara):
        mascara[rng.integers(0, len(mascara))] = True
    return mascara

def seleccion_torneo(poblacion, aptitudes, tamano_torneo, rng):
    indices = rng.integers(0, len(poblacion), size=tamano_torneo)
    mejor_idx = indices[np.argmax(aptitudes[indices])]
    return poblacion[mejor_idx].copy()

def cruce_uniforme(padre1, padre2, prob_cruce, rng):
    if rng.random() >= prob_cruce:
        return padre1.copy(), padre2.copy()
    mascara_cruce = rng.random(len(padre1)) < 0.5
    hijo1 = np.where(mascara_cruce, padre1, padre2)
    hijo2 = np.where(mascara_cruce, padre2, padre1)
    return hijo1, hijo2

def mutacion_bit_flip(individuo, prob_mutacion, rng):
    mutaciones = rng.random(len(individuo)) < prob_mutacion
    individuo = np.logical_xor(individuo, mutaciones)
    return asegurar_mascara_no_vacia(individuo, rng)

def algoritmo_genetico_binario(
    X_entrenamiento,
    y_entrenamiento,
    X_val,
    y_val,
    X_prueba,
    y_prueba,
    config,
    semilla,
    alpha=None,
):
    inicio = time.perf_counter()
    alpha_efectivo = config.alpha_base if alpha is None else alpha
    rng = np.random.default_rng(semilla)
    num_caracteristicas = X_entrenamiento.shape[1]

    poblacion = [
        asegurar_mascara_no_vacia(rng.random(num_caracteristicas) < 0.5, rng)
        for _ in range(config.tamano_poblacion)
    ]

    cache_aptitud = {}

    def calcular_aptitud(mascara):
        clave = mascara.tobytes()
        if clave not in cache_aptitud:
            f1_val = evaluar_mascara(
                X_entrenamiento, y_entrenamiento, X_val, y_val, mascara
            )
            proporcion = float(np.sum(mascara)) / num_caracteristicas
            aptitud = (1.0 - alpha_efectivo) * f1_val - alpha_efectivo * proporcion
            cache_aptitud[clave] = aptitud
        return cache_aptitud[clave]

    historial_mejor_aptitud = []
    mejor_mascara = None
    mejor_aptitud = -np.inf

    for _ in range(config.generaciones):
        aptitudes = np.array([calcular_aptitud(ind) for ind in poblacion])
        idx_elite = int(np.argmax(aptitudes))

        if aptitudes[idx_elite] > mejor_aptitud:
            mejor_aptitud = float(aptitudes[idx_elite])
            mejor_mascara = poblacion[idx_elite].copy()

        historial_mejor_aptitud.append(mejor_aptitud)

        nueva_poblacion = [poblacion[idx_elite].copy()]
        while len(nueva_poblacion) < config.tamano_poblacion:
            p1 = seleccion_torneo(poblacion, aptitudes, config.tamano_torneo, rng)
            p2 = seleccion_torneo(poblacion, aptitudes, config.tamano_torneo, rng)
            h1, h2 = cruce_uniforme(p1, p2, config.prob_cruce, rng)
            h1 = asegurar_mascara_no_vacia(mutacion_bit_flip(h1, config.prob_mutacion, rng), rng)
            h2 = asegurar_mascara_no_vacia(mutacion_bit_flip(h2, config.prob_mutacion, rng), rng)
            nueva_poblacion.append(h1)
            if len(nueva_poblacion) < config.tamano_poblacion:
                nueva_poblacion.append(h2)

        poblacion = nueva_poblacion

    f1_prueba = evaluar_mascara(
        X_entrenamiento, y_entrenamiento, X_prueba, y_prueba, mejor_mascara
    )
    tiempo_s = time.perf_counter() - inicio

    return {
        "mascara": mejor_mascara,
        "f1_prueba": f1_prueba,
        "mejor_aptitud_val": mejor_aptitud,
        "num_variables": int(np.sum(mejor_mascara)),
        "tiempo_s": tiempo_s,
        "evaluaciones": len(cache_aptitud),
        "historial_aptitud": historial_mejor_aptitud,
    }

### Función por semilla y prueba piloto técnica

La siguiente celda define `ejecutar_semilla`, la cual aplica el protocolo completo sobre una semilla dada: genera la partición sin fuga de datos, ejecuta primero el algoritmo genético binario, utiliza el número exacto de variables seleccionadas $k$ para ejecutar la selección univariada en igualdad de condiciones y finalmente evalúa el modelo con todas las variables. Cada medición se estructura en un diccionario compatible con el esquema de registro del proyecto.

Al final de la celda se ejecuta una prueba piloto con la semilla `0` para verificar el correcto funcionamiento de los tres métodos, comprobar los tiempos de ejecución y validar la coherencia de las métricas antes de lanzar las diez semillas completas.

In [10]:
def ejecutar_semilla(X, y, semilla, config, alpha=None, prefijo_ejecucion=1):
    alpha_efectivo = config.alpha_base if alpha is None else alpha
    X_ent, X_val, X_pru, y_ent, y_val, y_pru = dividir_datos(
        X, y, semilla, config.proporcion_val, config.proporcion_prueba
    )

    res_ag = algoritmo_genetico_binario(
        X_ent, y_ent, X_val, y_val, X_pru, y_pru, config, semilla, alpha=alpha_efectivo
    )
    k_seleccionado = res_ag["num_variables"]

    res_uni = evaluar_seleccion_univariada(X_ent, y_ent, X_pru, y_pru, k_seleccionado)
    res_todas = evaluar_todas_variables(X_ent, y_ent, X_pru, y_pru)

    registros = [
        {
            "ejecucion": f"{prefijo_ejecucion}_AG",
            "metodo": f"AG_alpha_{alpha_efectivo}",
            "instancia": "breast_cancer",
            "semilla": semilla,
            "metrica_principal": res_ag["f1_prueba"],
            "num_variables": res_ag["num_variables"],
            "tiempo_s": res_ag["tiempo_s"],
            "evaluaciones": res_ag["evaluaciones"],
            "observaciones": f"fitness_val={res_ag['mejor_aptitud_val']:.4f}",
        },
        {
            "ejecucion": f"{prefijo_ejecucion}_UNI",
            "metodo": f"Univariada_k_AG_{alpha_efectivo}",
            "instancia": "breast_cancer",
            "semilla": semilla,
            "metrica_principal": res_uni["f1_prueba"],
            "num_variables": res_uni["num_variables"],
            "tiempo_s": res_uni["tiempo_s"],
            "evaluaciones": res_uni["evaluaciones"],
            "observaciones": f"k_igualado={k_seleccionado}",
        },
        {
            "ejecucion": f"{prefijo_ejecucion}_TODAS",
            "metodo": "Todas_las_variables",
            "instancia": "breast_cancer",
            "semilla": semilla,
            "metrica_principal": res_todas["f1_prueba"],
            "num_variables": res_todas["num_variables"],
            "tiempo_s": res_todas["tiempo_s"],
            "evaluaciones": res_todas["evaluaciones"],
            "observaciones": "linea_base_30_vars",
        },
    ]

    detalles = {
        "mascara_ag": res_ag["mascara"],
        "mascara_uni": res_uni["mascara"],
        "historial_aptitud": res_ag["historial_aptitud"],
    }
    return registros, detalles

registros_piloto, detalles_piloto = ejecutar_semilla(
    X_total, y_total, semilla=0, config=config, alpha=config.alpha_base
)

for fila in registros_piloto:
    print(
        f"{fila['metodo']} | F1 prueba: {fila['metrica_principal']:.4f} | "
        f"Variables: {fila['num_variables']} | Tiempo: {fila['tiempo_s']:.3f}s | "
        f"Evaluaciones: {fila['evaluaciones']}"
    )

AG_alpha_0.05 | F1 prueba: 0.9521 | Variables: 6 | Tiempo: 4.198s | Evaluaciones: 798
Univariada_k_AG_0.05 | F1 prueba: 0.9343 | Variables: 6 | Tiempo: 0.005s | Evaluaciones: 1
Todas_las_variables | F1 prueba: 0.9716 | Variables: 30 | Tiempo: 0.005s | Evaluaciones: 1



## 5. Experimento principal: evaluación con diez semillas

### Objetivo

Evaluar el comportamiento del algoritmo genético binario frente a la selección univariada y al modelo que utiliza todas las características, mediante diez ejecuciones reproducibles.

### Metodología

Se utilizará el conjunto de datos Breast Cancer Wisconsin (Diagnostic), con 569 observaciones y 30 características numéricas.

Para cada semilla del 0 al 9 se realizará:

1. Una división estratificada de los datos en entrenamiento (60 %), validación (20 %) y prueba (20 %).
2. Una búsqueda evolutiva mediante el algoritmo genético binario, con alpha = 0.05.
3. Una evaluación mediante SelectKBest, utilizando el mismo número de características seleccionado por el algoritmo genético.
4. Una evaluación de referencia con las 30 características originales.
5. El registro del F1 macro de prueba, número de variables seleccionadas, tiempo de ejecución y cantidad de evaluaciones.

El algoritmo genético utilizará exclusivamente entrenamiento y validación para seleccionar las características. El conjunto de prueba se reservará para la evaluación final de cada ejecución.

### Reproducibilidad

Se utilizarán diez semillas: 0, 1, 2, 3, 4, 5, 6, 7, 8 y 9.

La configuración evolutiva permanece fija: población de 30 individuos, 30 generaciones, torneo de tamaño 3, probabilidad de cruce de 0.8 y probabilidad de mutación de 0.05.

### Resultados esperados

El experimento producirá 30 registros: diez por cada método. Se conservarán las máscaras binarias seleccionadas y los historiales de aptitud para posteriores análisis de estabilidad y convergencia.

La reducción del número de características y la conservación del desempeño predictivo se evaluarán a partir de los resultados obtenidos, sin asumir de antemano que el algoritmo genético será superior.


In [11]:
# 5.1 EJECUCION DEL EXPERIMENTO PRINCIPAL

# Carpeta para guardar los resultados
carpeta_resultados = Path("resultados_tema9")
carpeta_resultados.mkdir(exist_ok=True)

# Listas para almacenar los resultados
resultados_base = []
detalles_base = {}

# Valor de alpha definido en el protocolo
alpha_base = config.alpha_base

# Ejecutar las diez semillas
for semilla in config.semillas:

    print(f"\nEjecutando semilla {semilla}...")

    # Ejecutar algoritmo genetico, univariada y modelo base
    registros, detalles = ejecutar_semilla(
        X_total,
        y_total,
        semilla=semilla,
        config=config,
        alpha=alpha_base,
        prefijo_ejecucion=semilla + 1
    )

    # Guardar los tres registros de esta semilla
    resultados_base.extend(registros)

    # Guardar las mascaras y el historial
    detalles_base[semilla] = detalles

    # Exportar CSV actualizado
    df_base = pd.DataFrame(resultados_base)
    df_base.to_csv(
        carpeta_resultados / "experimento_base.csv",
        index=False
    )

    # Guardar los detalles numericos de esta semilla
    np.savez_compressed(
        carpeta_resultados / f"detalles_semilla_{semilla}.npz",
        mascara_ag=detalles["mascara_ag"],
        mascara_uni=detalles["mascara_uni"],
        historial_aptitud=detalles["historial_aptitud"]
    )

    print(f"Semilla {semilla} completada.")
    for fila in registros:
        print(
            f"  {fila['metodo']}: "
            f"F1={fila['metrica_principal']:.4f}, "
            f"variables={fila['num_variables']}, "
            f"tiempo={fila['tiempo_s']:.2f}s"
        )

print("\nEXPERIMENTO PRINCIPAL FINALIZADO")
print(f"Total de registros: {len(resultados_base)}")
print("Resultados guardados en:", carpeta_resultados)



Ejecutando semilla 0...
Semilla 0 completada.
  AG_alpha_0.05: F1=0.9521, variables=6, tiempo=4.29s
  Univariada_k_AG_0.05: F1=0.9343, variables=6, tiempo=0.01s
  Todas_las_variables: F1=0.9716, variables=30, tiempo=0.01s

Ejecutando semilla 1...
Semilla 1 completada.
  AG_alpha_0.05: F1=0.9716, variables=7, tiempo=4.39s
  Univariada_k_AG_0.05: F1=0.9623, variables=7, tiempo=0.01s
  Todas_las_variables: F1=0.9810, variables=30, tiempo=0.01s

Ejecutando semilla 2...
Semilla 2 completada.
  AG_alpha_0.05: F1=0.9615, variables=6, tiempo=4.40s
  Univariada_k_AG_0.05: F1=0.9716, variables=6, tiempo=0.01s
  Todas_las_variables: F1=0.9716, variables=30, tiempo=0.01s

Ejecutando semilla 3...
Semilla 3 completada.
  AG_alpha_0.05: F1=0.9810, variables=10, tiempo=4.66s
  Univariada_k_AG_0.05: F1=0.9716, variables=10, tiempo=0.01s
  Todas_las_variables: F1=0.9810, variables=30, tiempo=0.01s

Ejecutando semilla 4...
Semilla 4 completada.
  AG_alpha_0.05: F1=0.9719, variables=6, tiempo=4.21s
  Uni